<a href="https://colab.research.google.com/github/macrojocodes/foundry-f24-fundme/blob/main/MovieReviewPredictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Cell 1: Install dependencies ---
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [2]:
# --- Cell 2: Imports ---
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [5]:
import pandas as pd
import zipfile
from sklearn.model_selection import train_test_split
from datasets import Dataset

# --- Cell 3: Load IMDB dataset from CSV ---

# Define the path to the zipped CSV file
zip_file_path = "/IMDB Dataset.csv.zip"

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(".")

# The extracted file will likely be named 'IMDB Dataset.csv'
csv_file_path = "IMDB Dataset.csv"

# Load the CSV into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Convert sentiment to numerical labels (0 for negative, 1 for positive)
df['sentiment'] = df['sentiment'].map({'negative': 0, 'positive': 1})

# Split the DataFrame into training and testing sets
# We'll use stratify to ensure an even distribution of sentiment in both sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])

# Reset indices after splitting
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Apply shuffle and select ranges as in the original code, but on the new datasets
train_dataset = train_dataset.shuffle(seed=42).select(range(4000))
test_dataset = test_dataset.shuffle(seed=42).select(range(1000))

# Rename the 'sentiment' column to 'label' for compatibility with Trainer
train_dataset = train_dataset.rename_column("sentiment", "label")
test_dataset = test_dataset.rename_column("sentiment", "label")

print(f"Training dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Training dataset size: 4000
Test dataset size: 1000


In [8]:
# --- Cell 4: Tokenize ---
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["review"], padding="max_length", truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
# --- Cell 5: Load pretrained DistilBERT for binary classification ---
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
# --- Cell 6: Define metrics ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [11]:
# --- Cell 7: Training setup ---
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


In [12]:
# --- Cell 8: Fine-tune ---
try:
    trainer.train()
except ImportError as e:
    if "VideoReader" in str(e) and "torchvision.io" in str(e):
        print("Encountered an ImportError related to `torchvision.io.VideoReader`.")
        print("This often indicates a version mismatch or missing `torchvision` package.")
        print("Attempting to upgrade `torch` and `torchvision`. Please restart the runtime and re-run all cells after this.")
        !pip install --upgrade torch torchvision
    else:
        raise e

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.312315,0.268504,0.890000,0.877323,0.914729,0.895636
2,0.178787,0.256040,0.910000,0.911197,0.914729,0.912959


In [13]:
# --- Cell 9: Final evaluation ---
metrics = trainer.evaluate()
print("\nFinal evaluation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.178787,0.256040,2,0.910000,0.911197,0.914729,0.912959



Final evaluation metrics:
  eval_loss: 0.256039559841156
  eval_accuracy: 0.91
  eval_precision: 0.9111969111969112
  eval_recall: 0.9147286821705426
  eval_f1: 0.9129593810444874


In [14]:
sample_reviews = [
    "This movie was a complete waste of time, terrible acting and a boring plot.",
    "Absolutely loved it! The performances were brilliant and the story kept me hooked.",
]

inputs = tokenizer(sample_reviews, padding=True, truncation=True, max_length=256, return_tensors="pt")
model.eval()
with torch.no_grad():
    outputs = model(**{k: v.to(model.device) for k, v in inputs.items()})
    preds = torch.argmax(outputs.logits, dim=-1)

for review, pred in zip(sample_reviews, preds):
    label = "Positive" if pred.item() == 1 else "Negative"
    print(f"\nReview: {review}\nPredicted sentiment: {label}")


Review: This movie was a complete waste of time, terrible acting and a boring plot.
Predicted sentiment: Negative

Review: Absolutely loved it! The performances were brilliant and the story kept me hooked.
Predicted sentiment: Positive
